In [ ]:
%load_ext autoreload
%autoreload 2
%reset -f

# Imports

In [ ]:
from locallib.picarrodb import *
from locallib.query import *
from locallib.box import *
from locallib.pandas import *

import pandas as pd
import geopandas as gpd
import contextily as ctx
import pandas as pd
import os

import sys
import os

%matplotlib widget
import matplotlib.pyplot as plt
from shapely import wkt
from shapely.ops import unary_union
import matplotlib.pyplot as plt
from h3 import *
from shapely.geometry import Point
from shapely.geometry import LineString, Point
import numpy as np

## Query the surveys

In [ ]:
a = get_reports('Cadent',years = [2026]).execute([EU2_Conn])
report_bc = a.iloc[[20]].copy()

## Query the geometries

In [ ]:
report_bc.db.set_query(query_SurveyH3Aggregation_byReport(report_table = '#TempReport'))
agg_segments =report_bc.db.execute([EU2_Conn], source_col = 'ReportId', temp_table_name = '#TempReport')
report_bc.db.set_query(query_Segments_byReport(report_table = '#TempReport'))
segments = report_bc.db.execute([EU2_Conn], source_col = 'ReportId', temp_table_name = '#TempReport')

## Isolate a single survey

In [ ]:
import osmnx as ox

In [ ]:
l = 0
surveys = segments['SurveyId'].unique()
survey = segments[segments['SurveyId'] == surveys[l]]
#survey = segments
survey['Breadcrumb_wkt'] = survey['Breadcrumb'].apply(wkt.loads)
survey_gdf = gpd.GeoDataFrame(survey, geometry = 'Breadcrumb_wkt', crs = 'EPSG:4326')
utm_crs = survey_gdf.estimate_utm_crs()
survey_gdf = survey_gdf.to_crs(utm_crs)

In [ ]:
contour = agg_segments[agg_segments['SurveyId'] == surveys[l]]
contour['Breadcrumb_wkt'] = contour['Breadcrumb'].apply(wkt.loads)
contour_gdf = gpd.GeoDataFrame(contour, geometry = 'Breadcrumb_wkt', crs = 'EPSG:4326')
contour_gdf = contour_gdf.to_crs(utm_crs)
edge_gdf = contour_gdf.geometry.buffer(5, cap_style=2)
import osmnx as ox

edge_gdf_4326 = edge_gdf.to_crs('EPSG:4326')

# Convert the GeoSeries to a single MultiPolygon, if necessary
if hasattr(edge_gdf_4326, "unary_union"):
    geom = edge_gdf_4326.unary_union
else:
    geom = edge_gdf_4326[0]

# Defensive: Check if ox.settings.overpass_settings is None and set to default if so
if getattr(ox.settings, "overpass_settings", None) is None:
    # Default value from OSMnx 1.1.2 (may vary for other versions)
    ox.settings.overpass_settings = "[out:json][timeout:{timeout}]{maxsize};"

try:
    G = ox.graph_from_polygon(geom)
except AttributeError as e:
    print("Error: ox.settings.overpass_settings is None or misconfigured.")
    print(e)
    G = None

# Query the OSM data